
# Inferencia en producción y enriquecimiento de etiquetas reales

**Autor**: Juan Carlos Alfaro Jiménez

Esta libreta implementa dos responsabilidades complementarias que cierran el ciclo de monitorización del modelo en producción.

La primera es la **inferencia en lote**: carga el *pipeline* `Spark MLlib` registrado bajo el alias `champion` en `Unity Catalog`, lee las transacciones que han llegado desde la última ejecución y que aún no tienen predicción en `gold_fraud_inference_enriched`, las enriquece con las características del cliente mediante el `PiT` *join* del *feature store* y las transforma con `.transform()` de forma distribuida sobre el clúster de `Databricks`.

La segunda es el **enriquecimiento de etiquetas**: los fraudes se confirman con retraso una vez que el equipo de revisión cierra cada caso. Cuando una etiqueta real llega a `bronze_labels`, se propaga a `gold_fraud_inference_enriched` mediante un `MERGE` incremental idempotente. Esta tabla, que combina las características de cada transacción, la predicción del modelo y la etiqueta real confirmada, es el dato de entrada de `Databricks Lakehouse Monitoring` para calcular métricas de rendimiento y equidad en producción.


## 1. Importaciones y configuración

In [0]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
exec(open("07_Utils.py").read(), globals())

Total rows: 10,356,715
Total columns: 39

Semantic version: 3

Train period: 2023-12-30 → 2024-12-30
Validation period: 2024-12-31 → 2025-04-30
Test period: 2025-05-01 → 2025-06-30

Train rows: 6,286,651
Validation rows: 2,620,969
Test rows: 1,382,825

Numeric (21): ['cart_value', 'age', 'return_rate', 'count_events_1h', 'sum_cart_value_1h', 'avg_cart_value_1h', 'distinct_categories_1h', 'count_sessions_24h', 'sum_cart_value_24h', 'avg_cart_value_24h', 'max_cart_value_24h', 'distinct_categories_24h', 'count_add_to_cart_24h', 'count_events_7d', 'sum_cart_value_7d', 'distinct_categories_7d', 'count_events_30d', 'sum_cart_value_30d', 'avg_cart_value_30d', 'num_abandoned_confirmed_30d', 'cart_value_24h_vs_avg_30d_ratio']
Boolean (3): ['has_app_installed', 'email_opt_in', 'push_opt_in']
Categorical (9): ['item_category', 'event_type', 'user_type', 'gender', 'country', 'preferred_device', 'favourite_category', 'age_group', 'loyalty_segment']

Assembler inputs (34): ['cart_value_imp', 'age_im

In [0]:
exec(open("08_Utils.py").read(), globals())

08_Utils.py script loaded successfully.


In [0]:
exec(open("09_Utils.py").read(), globals())

Profile features (12): ['age', 'gender', 'country', 'user_type', 'preferred_device', 'favourite_category', 'age_group', 'loyalty_segment', 'return_rate', 'has_app_installed', 'email_opt_in', 'push_opt_in']
Aggregation features (18): ['count_events_1h', 'sum_cart_value_1h', 'avg_cart_value_1h', 'distinct_categories_1h', 'count_sessions_24h', 'sum_cart_value_24h', 'avg_cart_value_24h', 'max_cart_value_24h', 'distinct_categories_24h', 'count_add_to_cart_24h', 'count_events_7d', 'sum_cart_value_7d', 'distinct_categories_7d', 'count_events_30d', 'sum_cart_value_30d', 'avg_cart_value_30d', 'num_abandoned_confirmed_30d', 'cart_value_24h_vs_avg_30d_ratio']
Total feature columns: 30

09_Utils.py script loaded successfully.


In [0]:
from datetime import timedelta

from databricks.feature_engineering import FeatureEngineeringClient

from delta.tables import DeltaTable

import mlflow
import mlflow.spark
from mlflow import MlflowClient

from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

In [0]:
notebook_path_raw = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook = Path(notebook_path_raw).name

print(f"Project: {project}, team: {team}, environment: {environment}")
print(f"Notebook: {notebook}")
print(f"Champion model: {uc_model_name}")
print(f"Spine table: {spine_table}")
print(f"Inference enriched table: {inference_enriched_table}")
print(f"Abandonment labels table: {fraud_labels_table}")

Project: cart_abandonment_prediction, team: ml_engineering, environment: development
Notebook: 09_Inference_And_Label_Enrichment
Champion model: workspace.abandono_carrito_comercio_electronico.abandonment_lr_pipeline
Spine table: workspace.abandono_carrito_comercio_electronico.gold_abandonment_spine
Inference enriched table: workspace.abandono_carrito_comercio_electronico.gold_abandonment_inference_enriched
Abandonment labels table: workspace.abandono_carrito_comercio_electronico.bronze_labels



## 2. Carga del modelo `champion`

Se carga el *pipeline* `Spark MLlib` registrado bajo el alias `champion` en `Unity Catalog`. La versión concreta se persiste en `gold_fraud_inference_enriched` para garantizar la trazabilidad completa entre cada predicción y el modelo que la generó.

In [0]:
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()
champion_version = client.get_model_version_by_alias(name = uc_model_name, alias = "champion")
champion_model_version = champion_version.version

pipeline_model = mlflow.spark.load_model(f"models:/{uc_model_name}@champion")

print(f"Champion model loaded: {uc_model_name}")
print(f"Version: {champion_model_version}")

Champion model loaded: workspace.abandono_carrito_comercio_electronico.abandonment_lr_pipeline
Version: 2



## 3. Inicialización de la tabla de inferencias enriquecidas

La tabla `gold_fraud_inference_enriched` debe existir antes de poder leer las transacciones ya puntuadas en la sección siguiente. Si es la primera ejecución, se crea vacía con el esquema correcto derivado de un único ejemplo de predicción. En ejecuciones posteriores esta sección no tiene ningún efecto.

Una vez creada, hay que configurar el monitor de `Databricks Lakehouse Monitoring` desde el `Catalog Explorer` siguiendo los pasos descritos en la sección de conclusiones antes de programar la ejecución automática de esta libreta.

In [0]:
table_exists = spark.catalog.tableExists(inference_enriched_table)

if not table_exists:
    (
        spark.table(baseline_table_name)
             .limit(0)
             .write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(inference_enriched_table)
    )

    spark.sql(
        f"ALTER TABLE {inference_enriched_table} "
        f"SET TBLPROPERTIES ("
        f"'delta.enableChangeDataFeed' = 'true', "
        f"'project' = '{project}', "
        f"'team' = '{team}')"
    )

    spark.sql(f"""
        ALTER TABLE {inference_enriched_table}
        SET TAGS (
            'project' = '{project}',
            'team' = '{team}'
        )
    """)

    spark.sql(
        f"COMMENT ON TABLE {inference_enriched_table} IS "
        f"'Production inference table. Contains raw transaction features, "
        f"model predictions and confirmed fraud labels. "
        f"Monitored by `Databricks Lakehouse Monitoring` against `gold_fraud_test_baseline`.'"
    )

    print(f"Table created: {inference_enriched_table}")
else:
    print(f"Table already exists: {inference_enriched_table}")

Table created: workspace.abandono_carrito_comercio_electronico.gold_abandonment_inference_enriched



## 4. Lectura de transacciones pendientes de predicción

Las transacciones de producción son aquellas posteriores al `champion_test_end_date`, la fecha máxima de datos que vio el `champion` actual durante alguna de sus fases (entrenamiento, validación o prueba). Este valor se lee de la etiqueta `test_end_date` de la versión del `champion` en `Unity Catalog`, escrita en `08_Production.ipynb` durante la promoción. Se usa esta fecha en lugar de recalcularla desde `gold_fraud_training_dataset` para blindar el *pipeline* contra posibles descoordinaciones: si la tabla de entrenamiento se actualiza antes de que termine un ciclo completo de reentrenamiento, el valor recalculado quedaría adelantado respecto al que realmente vio el `champion`, dejando transacciones sin puntuar. Se filtra `gold_fraud_inference_spine` por esa fecha y se excluyen mediante un `LEFT ANTI JOIN` las transacciones que ya tienen predicción en `gold_fraud_inference_enriched`. La operación es idempotente: si la libreta falla y se relanza no se duplican predicciones.

In [0]:
# --- CELDA 4 AJUSTADA PARA DETECTAR LA SIMULACIÓN ---

# Comentamos el cálculo automático para forzar la fecha de la prueba
# champion_test_end_date = champion_version.tags.get("test_end_date")

# Forzamos la fecha de inicio de producción al 29 de junio de 2025
production_start_date = "2025-06-29"

# Identificamos qué transacciones ya han sido puntuadas anteriormente 
# para no repetir trabajo (idempotencia)
already_scored_df = (
    spark.table(inference_enriched_table)
         .select(transaction_id_column)
)

# Filtramos la 'spine' para coger solo lo nuevo desde el 29 de junio
new_spine_df = (
    spark.table(spine_table)
         .filter(F.col(date_column) >= production_start_date)
         .join(already_scored_df, on = transaction_id_column, how = "left_anti")
)

n_new = new_spine_df.count()

print(f"Production cutoff date (Manual override): {production_start_date}")
print(f"New transactions pending prediction: {n_new:,}")

Production cutoff date (Manual override): 2025-06-29
New transactions pending prediction: 364,020



## 5. Enriquecimiento con el *feature store*

Las características del cliente no viajan con la transacción: en producción se recuperan en tiempo real desde el *online feature store*. En este entorno se simulan mediante un `PiT` *join* contra `gold_customer_profile` y `gold_customer_aggregations_inference`, replicando exactamente el enriquecimiento que realizó `05_Training_Dataset_Generation` durante el entrenamiento. Esto garantiza que no hay *training-serving skew*: el *pipeline* recibe exactamente el mismo conjunto de características con el que fue ajustado.

In [0]:
fe = FeatureEngineeringClient()

inference_set = fe.create_training_set(
    df = new_spine_df,
    feature_lookups = feature_lookups,
    label = None
)

new_transactions_df = inference_set.load_df()

print(f"Enriched transactions: {new_transactions_df.count():,}")

Enriched transactions: 364,020



## 6. Predicción e inserción en la tabla enriquecida

Se aplica `.transform()` del *pipeline* `Spark MLlib` sobre el `DataFrame` de transacciones nuevas. Se extrae la probabilidad de fraude como columna escalar desde el vector de probabilidades, y se añaden el instante de inferencia y la versión del modelo. La columna `is_fraud` se fuerza a nulo porque la etiqueta real no es conocida en el momento de la predicción.

El `MERGE` inserta únicamente las filas cuyo `transaction_id` no existe aún en la tabla enriquecida.

In [0]:
if n_new > 0:
    # 1. Generamos las predicciones base
    scored_df = (
        pipeline_model
        .transform(new_transactions_df)
        .withColumn(
            "prob_abandonment",
            vector_to_array(F.col(probability_column)).getItem(1)
        )
        .withColumn(inference_timestamp_col, F.current_timestamp())
        .withColumn(model_version_col, F.lit(champion_model_version))
        .withColumn(label_column, F.lit(None).cast("long"))
        .withColumn(prediction_column, F.col(prediction_column).cast("long"))
    )

    # 2. ALINEACIÓN DE ESQUEMA AUTOMÁTICA
    # Leemos las columnas que tiene la tabla física en el catálogo
    target_table = spark.table(inference_enriched_table)
    target_schema = target_table.schema
    
    # Buscamos qué columnas faltan en nuestras predicciones
    for field in target_schema:
        if field.name not in scored_df.columns:
            # Añadimos la columna faltante como Nula con su tipo de dato correcto
            scored_df = scored_df.withColumn(field.name, F.lit(None).cast(field.dataType))
            print(f"⚠️ Columna missing añadida como Null: {field.name}")

    # 3. Reordenamos las columnas para que coincidan exactamente con la tabla
    scored_df = scored_df.select(target_table.columns)

    # 4. Ejecutamos el MERGE
    (
        DeltaTable.forName(spark, inference_enriched_table)
                  .alias("target")
                  .merge(
                      scored_df.alias("source"),
                      f"target.{transaction_id_column} = source.{transaction_id_column}"
                  )
                  .whenNotMatchedInsertAll()
                  .execute()
    )

    # Métricas finales
    n_fraud_pred = scored_df.filter(F.col(prediction_column) == 1).count()
    n_legit_pred = scored_df.filter(F.col(prediction_column) == 0).count()

    print(f"✅ Proceso completado. {n_new:,} predicciones insertadas.")
    print(f"Detalle: {n_fraud_pred} Abandonos, {n_legit_pred} Ventas.")
else:
    print("No new transactions to score.")

⚠️ Columna missing añadida como Null: registration_date
⚠️ Columna missing añadida como Null: __END_AT
✅ Proceso completado. 364,020 predicciones insertadas.
Detalle: 0 Abandonos, 0 Ventas.


In [0]:
# Consulta para verificar las predicciones reales en la tabla
df_check = spark.table(inference_enriched_table)
display(df_check.select("session_id", "prediction", "prob_abandonment").limit(20))

# Conteo real fuera de las variables del script
print(f"Total registros: {df_check.count():,}")
print(f"Abandonos detectados (1): {df_check.filter('prediction = 1').count():,}")
print(f"Ventas detectadas (0): {df_check.filter('prediction = 0').count():,}")

session_id,prediction,prob_abandonment
SESF19F9A66F728,1,0.46195858514630905
SESF19F9A66F728,1,0.46195858514630905
SESF19F9A66F728,1,0.46195858514630905
SESF19F9A66F728,1,0.46195858514630905
SES76AF19181DCA,1,0.6423889380095483
SES76AF19181DCA,1,0.6423889380095483
SES76AF19181DCA,1,0.6423889380095483
SES76AF19181DCA,1,0.6423889380095483
SES2C442A80CE75,1,0.5253428746721662
SES2C442A80CE75,1,0.5253428746721662


Total registros: 364,020
Abandonos detectados (1): 357,770
Ventas detectadas (0): 6,250



## 7. Propagación de etiquetas reales

Se leen de `bronze_labels` únicamente las etiquetas de transacciones de producción (posteriores a `production_start_date`) cuyo `transaction_id` no tiene todavía una etiqueta confirmada en `gold_fraud_inference_enriched`. El resultado esperado es cero etiquetas pendientes en las primeras ejecuciones, ya que los fraudes se confirman con retraso una vez que el equipo de revisión cierra cada caso. El `MERGE` actualiza `is_fraud` en las filas que ya existen en la tabla. Las filas sin etiqueta confirmada permanecen con `is_fraud` nulo hasta que se confirmen en una ejecución posterior.

In [0]:
already_labelled_df = (
    DeltaTable.forName(spark, inference_enriched_table)
              .toDF()
              .filter(F.col(label_column).isNotNull())
              .select(transaction_id_column)
)

pending_labels_df = (
    spark.table(fraud_labels_table)
         .filter(F.col(label_available_date_column) >= production_start_date)
         .join(already_labelled_df, on = transaction_id_column, how = "left_anti")
         .select(transaction_id_column, label_column)
)

n_pending_total = pending_labels_df.count()
n_pending_fraud = pending_labels_df.filter(F.col(label_column) == 1).count()
n_pending_legit = pending_labels_df.filter(F.col(label_column) == 0).count()

print(f"Abandonment labels pending propagation: {n_pending_total:,}")
print(f"Abandono (Label 1): {n_pending_fraud:,}")
print(f"Venta / Éxito (Label 0): {n_pending_legit:,}")

Abandonment labels pending propagation: 190,279
Abandono (Label 1): 115,255
Venta / Éxito (Label 0): 75,024


In [0]:
# 1. Limpiamos los duplicados del origen (source)
# Nos quedamos solo con una entrada por sesión (la más reciente si fuera necesario)
labels_deduplicated_df = (
    pending_labels_df
    .dropDuplicates([transaction_id_column]) 
)

# 2. Ejecutamos el MERGE con el DataFrame limpio
(
    DeltaTable.forName(spark, inference_enriched_table)
              .alias("target")
              .merge(
                  labels_deduplicated_df.alias("source"),
                  f"target.{transaction_id_column} = source.{transaction_id_column}"
              )
              .whenMatchedUpdate(
                  set = {f"target.{label_column}": f"source.{label_column}"}
              )
              .execute()
)

n_propagated = labels_deduplicated_df.count()
print(f"✅ Éxito: Se han propagado {n_propagated:,} etiquetas únicas sin conflictos.")

✅ Éxito: Se han propagado 4,135 etiquetas únicas sin conflictos.


In [0]:
# Consulta para ver cuántas veces acertó el modelo
display(spark.sql(f"""
  SELECT 
    will_abandon AS Realidad,
    prediction AS Prediccion,
    COUNT(*) as Total
  FROM {inference_enriched_table}
  WHERE will_abandon IS NOT NULL
  GROUP BY 1, 2
  ORDER BY 1, 2
"""))

Realidad,Prediccion,Total
0,0,3892
0,1,139940
1,0,2358
1,1,217830



## 8. Conclusiones y siguientes pasos

### ¿Qué hace esta libreta?

1. **Carga del modelo `champion`**: Recupera el pipeline `Spark MLlib` directamente desde el alias `champion` de `Unity Catalog` y la versión concreta para trazabilidad.
2. **Inicialización de la tabla**: Crea `gold_fraud_inference_enriched` vacía con el esquema correcto si es la primera ejecución, copiando la estructura de `gold_fraud_test_baseline` que garantiza compatibilidad con el monitor, y activa `delta.enableChangeDataFeed` para `Databricks Lakehouse Monitoring`.
3. **Lectura de transacciones pendientes**: Filtra `gold_fraud_inference_spine` para considerar únicamente las transacciones posteriores a `production_start_date` (el día siguiente a `test_end_date`, último dato visto por el modelo) e identifica mediante un `LEFT ANTI JOIN` las que todavía no tienen predicción en `gold_fraud_inference_enriched`. La operación es idempotente.
4. **Enriquecimiento con el *feature store***: Replica el `PiT` *join* de entrenamiento contra `gold_customer_profile` y `gold_customer_aggregations_inference` con las mismas `feature_names` definidas en `09_Utils.py`, garantizando que no hay *training-serving skew*.
5. **Inferencia en lote distribuida**: Aplica `.transform()` del pipeline `Spark MLlib` sobre el lote de transacciones nuevas e inserta el resultado en `gold_fraud_inference_enriched` con `is_fraud` nulo.
6. **Propagación de etiquetas**: Actualiza `is_fraud` en `gold_fraud_inference_enriched` para las transacciones cuya etiqueta real ha llegado a `bronze_labels`, filtrando también por `production_start_date`.

### ¿Por qué es necesaria?

`Databricks Lakehouse Monitoring` necesita las etiquetas reales para calcular métricas de rendimiento en producción (*AUC-PR*, *F1-score*, precisión, exhaustividad) y métricas de equidad sobre los grupos de interés. Sin este paso, el monitor solo puede calcular *data drift* de características pero no detectar degradación real del modelo ni sesgos en subgrupos.

### ¿Cuándo se ejecuta?

#### Primera ejecución (manual)

1. Ejecutar la libreta manualmente para que se cree e inicialice `gold_fraud_inference_enriched` con el esquema correcto.
2. Navegar a la tabla en `Catalog Explorer`, abrir la pestaña `Quality` y hacer clic en `Create monitor`.
3. Configurar el monitor con los siguientes parámetros: tipo de problema `Classification`, columna temporal `inference_timestamp`, identificador de modelo `model_version`, columna de predicción `prediction`, columna de etiqueta `is_fraud` y tabla de referencia `gold_fraud_test_baseline`.
4. Configurar las expresiones de *slice* como se detalla a continuación y hacer clic en `Create`.

#### Ejecuciones posteriores (automáticas)

Como tarea `Run_Inference_And_Label_Enrichment` en el trabajo `Credit Card Fraud Inference Pipeline`, que corre cada dos horas con un desfase de una hora respecto al trabajo `Credit Card Fraud Feature Pipeline`, para garantizar que las etiquetas confirmadas disponibles en `bronze_labels` se propaguen a `gold_fraud_inference_enriched` en cada ciclo con las características más recientes disponibles.

#### Expresiones de *slice* para el monitor

`Databricks Lakehouse Monitoring` calcula todas las métricas de rendimiento y *drift* tanto sobre el total de transacciones como, de forma independiente, sobre cada *slice* declarado. Esto permite detectar degradaciones que solo afectan a un subgrupo concreto aunque las métricas globales sigan siendo buenas.

Hay dos tipos de *slices* relevantes para este problema:

##### *Slices* operativos

Subgrupos definidos por características de la transacción que concentran la mayor parte del fraude. El monitor calcula métricas de rendimiento separadas para cada uno, lo que permite detectar si el modelo se degrada antes en las transacciones de mayor riesgo:

| Expresión | Justificación |
|---|---|
| `cross_border = 1` | Las transacciones transfronterizas tienen una tasa de fraude estructuralmente más alta. Una degradación en este subgrupo es especialmente costosa. |
| `is_tor_or_vpn = 1` | Las transacciones desde redes de anonimización son el subgrupo de mayor riesgo. Cualquier caída de rendimiento aquí es una señal de alerta crítica. |
| `three_ds_result = 'FAILED'` | Los intentos fallidos de autenticación 3-D Secure son un indicador fuerte de fraude intencional. |

##### *Slices* de equidad

Subgrupos definidos por atributos demográficos del cliente. `Databricks Lakehouse Monitoring` calcula automáticamente métricas de equidad (igualdad de oportunidades, paridad predictiva y paridad estadística) para cada uno cuando el tipo de problema es `Classification` y hay columna de etiqueta. El objetivo es detectar si el modelo genera tasas de falsos positivos desproporcionadas en algún grupo, lo que supondría que ciertos clientes tienen sus transacciones legítimas bloqueadas con más frecuencia que otros:

| Expresión | Justificación |
|---|---|
| `gender` | Detecta si la tasa de falsos positivos difiere sistemáticamente entre hombres, mujeres y otros géneros. |
| `age_group` | Detecta si el modelo es más agresivo bloqueando transacciones de clientes jóvenes o mayores. |
| `customer_segment` | Detecta si los clientes `standard` reciben más falsos positivos que los `vip` o `premium`, lo que indicaría un sesgo socioeconómico. |
| `country` | Detecta si hay disparidad geográfica en las métricas, especialmente relevante si la distribución de países en producción difiere de la del entrenamiento. |

Las métricas de equidad generadas automáticamente por el monitor para cada uno de estos *slices* son la **igualdad de oportunidades** (diferencia en tasa de verdaderos positivos entre grupos), la **paridad predictiva** (diferencia en precisión entre grupos) y la **paridad estadística** (diferencia en tasa de predicciones positivas entre grupos). Estas métricas están disponibles en la tabla `gold_fraud_inference_enriched_drift_metrics` generada por el monitor.